In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")

s18 = pd.read_csv(
    PROJECT_ROOT / "data/staging/luxury_tax/S18_Luxury_Tax.csv"
)

c04 = pd.read_csv(
    PROJECT_ROOT / "data/canonical/team_seasons.csv"
)

print("S18 SHAPE:", s18.shape)
print("S18 COLUMNS:", s18.columns.tolist())

print("\nDATA TYPES")
print(s18.dtypes)

print("\nMISSING VALUES")
print(s18.isna().sum())

print("\nEXACT DUPLICATE ROWS:", s18.duplicated().sum())

print("\nS18 SAMPLE")
display(s18.head(10))

print("\nC04 IDENTIFIER COLUMNS")
print([
    col for col in c04.columns
    if "team" in col.lower()
    or "franchise" in col.lower()
    or "season" in col.lower()
])

S18 SHAPE: (718, 11)
S18 COLUMNS: ['franchise_id', 'season', 'tax_threshold', 'taxable_payroll', 'tax_space', 'tax_paid', 'estimated_tax_bill', 'total_estimated_cost', 'active_salary', 'dead_salary', 'source_file']

DATA TYPES
franchise_id                str
season                      str
tax_threshold           float64
taxable_payroll         float64
tax_space               float64
tax_paid                float64
estimated_tax_bill      float64
total_estimated_cost    float64
active_salary           float64
dead_salary             float64
source_file                 str
dtype: object

MISSING VALUES
franchise_id              0
season                    0
tax_threshold            30
taxable_payroll         268
tax_space               268
tax_paid                450
estimated_tax_bill      624
total_estimated_cost    268
active_salary           268
dead_salary             302
source_file             268
dtype: int64

EXACT DUPLICATE ROWS: 0

S18 SAMPLE


,franchise_id,season,tax_threshold,taxable_payroll,tax_space,tax_paid,estimated_tax_bill,total_estimated_cost,active_salary,dead_salary,source_file
0,GSW,2002-2003,52900000.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1,LAC,2002-2003,52900000.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2,NYK,2002-2003,52900000.0,NaN,NaN,24370000.0,NaN,NaN,NaN,NaN,NaN
3,BRK,2002-2003,52900000.0,NaN,NaN,5730000.0,NaN,NaN,NaN,NaN,NaN
4,PHO,2002-2003,52900000.0,NaN,NaN,1890000.0,NaN,NaN,NaN,NaN,NaN
5,LAL,2002-2003,52900000.0,NaN,NaN,9750000.0,NaN,NaN,NaN,NaN,NaN
6,CLE,2002-2003,52900000.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
7,MIL,2002-2003,52900000.0,NaN,NaN,4730000.0,NaN,NaN,NaN,NaN,NaN
8,BOS,2002-2003,52900000.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
9,DAL,2002-2003,52900000.0,NaN,NaN,18500000.0,NaN,NaN,NaN,NaN,NaN



C04 IDENTIFIER COLUMNS
['team_name', 'season', 'franchise_id', 'team_payroll']


In [2]:
# Load the historical franchise aliases from C02.
aliases = pd.read_csv(
    PROJECT_ROOT / "data/canonical/franchise_season_aliases.csv"
)

print("S18 UNIQUE FRANCHISE CODES")
print(sorted(s18["franchise_id"].unique()))

print("\nS18 SEASON COVERAGE")
display(
    s18.groupby("season")
    .agg(
        rows=("franchise_id", "size"),
        actual_tax_records=("tax_paid", "count"),
        taxable_payroll_records=("taxable_payroll", "count"),
        estimated_tax_records=("estimated_tax_bill", "count")
    )
    .sort_index()
)

# Check whether S18 has one record per franchise code and season.
duplicate_keys = s18[
    s18.duplicated(["franchise_id", "season"], keep=False)
]

print("\nDUPLICATE S18 FRANCHISE-SEASON KEYS:", len(duplicate_keys))
display(duplicate_keys.head(10))

# First try the season-specific historical abbreviation.
s18_test = s18.merge(
    aliases[
        ["season", "team_abbreviation", "franchise_id"]
    ].rename(
        columns={
            "team_abbreviation": "s18_franchise_code",
            "franchise_id": "canonical_franchise_id"
        }
    ),
    left_on=["season", "franchise_id"],
    right_on=["season", "s18_franchise_code"],
    how="left",
    validate="many_to_one"
)

print("\nHISTORICAL ALIAS MATCHES")
print("Matched:", s18_test["canonical_franchise_id"].notna().sum())
print("Unmatched:", s18_test["canonical_franchise_id"].isna().sum())

print("\nUNMATCHED FRANCHISE CODES")
display(
    s18_test.loc[
        s18_test["canonical_franchise_id"].isna(),
        ["franchise_id", "season"]
    ].drop_duplicates()
    .sort_values(["franchise_id", "season"])
)

# Check how many matched S18 rows correspond to C04 keys.
matched_s18 = s18_test.dropna(
    subset=["canonical_franchise_id"]
).copy()

matched_s18["canonical_franchise_id"] = (
    matched_s18["canonical_franchise_id"].astype("int64")
)

c04_keys = c04[
    ["franchise_id", "season"]
].drop_duplicates()

compatibility = matched_s18.merge(
    c04_keys,
    left_on=["canonical_franchise_id", "season"],
    right_on=["franchise_id", "season"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

print("\nMATCHED S18 ROWS PRESENT IN C04:",
      (compatibility["_merge"] == "both").sum())

print("MATCHED S18 ROWS ABSENT FROM C04:",
      (compatibility["_merge"] == "left_only").sum())

S18 UNIQUE FRANCHISE CODES
['ATL', 'BOS', 'BRK', 'CHI', 'CHO', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']

S18 SEASON COVERAGE


,rows,actual_tax_records,taxable_payroll_records,estimated_tax_records
season,,,,
2002-2003,29,29,0,0
2003-2004,29,29,0,0
2004-2005,30,30,0,0
2005-2006,30,30,0,0
2006-2007,30,30,0,0
2007-2008,30,30,0,0
2008-2009,30,30,0,0
2009-2010,30,30,0,0
2010-2011,30,30,0,0



DUPLICATE S18 FRANCHISE-SEASON KEYS: 0


,franchise_id,season,tax_threshold,taxable_payroll,tax_space,tax_paid,estimated_tax_bill,total_estimated_cost,active_salary,dead_salary,source_file



HISTORICAL ALIAS MATCHES
Matched: 631
Unmatched: 87

UNMATCHED FRANCHISE CODES


,franchise_id,season
3,BRK,2002-2003
32,BRK,2003-2004
61,BRK,2004-2005
91,BRK,2005-2006
121,BRK,2006-2007
...,...,...
581,PHO,2021-2022
603,PHO,2022-2023
630,PHO,2023-2024
658,PHO,2024-2025



MATCHED S18 ROWS PRESENT IN C04: 631
MATCHED S18 ROWS ABSENT FROM C04: 0


In [3]:
# 1. Summarize unmatched S18 franchise codes.
unmatched = s18_test[
    s18_test["canonical_franchise_id"].isna()
].copy()

print("UNMATCHED S18 CODES")
display(
    unmatched.groupby("franchise_id")
    .agg(
        unmatched_rows=("season", "size"),
        first_season=("season", "min"),
        last_season=("season", "max")
    )
    .sort_values("unmatched_rows", ascending=False)
)

# 2. Inspect the alias table's structure.
print("\nALIAS COLUMNS")
print(aliases.columns.tolist())

# 3. Find every historical alias associated with unmatched codes.
problem_codes = unmatched["franchise_id"].unique()

print("\nAVAILABLE HISTORICAL ALIASES FOR THOSE CODES")
display(
    aliases[
        aliases["team_abbreviation"].isin(problem_codes)
    ][
        ["team_abbreviation", "franchise_id", "season"]
    ]
    .drop_duplicates()
    .groupby(["team_abbreviation", "franchise_id"])
    .agg(
        first_season=("season", "min"),
        last_season=("season", "max")
    )
    .reset_index()
    .sort_values("team_abbreviation")
)

UNMATCHED S18 CODES


,unmatched_rows,first_season,last_season
franchise_id,,,
BRK,24,2002-2003,2025-2026
PHO,24,2002-2003,2025-2026
CHO,22,2004-2005,2025-2026
NOP,11,2002-2003,2012-2013
OKC,6,2002-2003,2007-2008



ALIAS COLUMNS
['franchise_id', 'season', 'team_abbreviation', 'full_team_name', 'short_team_name']

AVAILABLE HISTORICAL ALIASES FOR THOSE CODES


,team_abbreviation,franchise_id,first_season,last_season
0,NOP,1610612740,2013-2014,2025-2026
1,OKC,1610612760,2008-2009,2025-2026


In [4]:
# Map S18's modern franchise codes to canonical franchise IDs.
# These five codes account for all 87 unmatched records.

code_fixes = {
    "BRK": 1610612751,  # Nets
    "PHO": 1610612756,  # Suns
    "CHO": 1610612766,  # Charlotte Bobcats/Hornets
    "NOP": 1610612740,  # New Orleans franchise
    "OKC": 1610612760,  # Seattle / Oklahoma City franchise
}

# Start with the successful historical alias matches.
s18_mapped = s18_test.copy()

# Fill only previously unmatched canonical franchise IDs.
missing_mask = s18_mapped["canonical_franchise_id"].isna()

s18_mapped.loc[
    missing_mask, "canonical_franchise_id"
] = (
    s18_mapped.loc[missing_mask, "franchise_id"]
    .map(code_fixes)
)

# Validate that every S18 record now maps.
assert s18_mapped["canonical_franchise_id"].notna().all(), (
    "Some S18 franchise codes remain unmapped."
)

s18_mapped["canonical_franchise_id"] = (
    s18_mapped["canonical_franchise_id"].astype("int64")
)

# Confirm uniqueness at the intended canonical grain.
assert not s18_mapped.duplicated(
    ["canonical_franchise_id", "season"]
).any(), "Duplicate canonical franchise-season keys found."

# Check all S18 keys exist in C04.
s18_key_check = s18_mapped.merge(
    c04[["franchise_id", "season"]],
    left_on=["canonical_franchise_id", "season"],
    right_on=["franchise_id", "season"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

assert (s18_key_check["_merge"] == "both").all(), (
    "Some mapped S18 records are absent from C04."
)

print("S18 MAPPING VALIDATION PASSED")
print("S18 records:", len(s18_mapped))
print("Mapped records:", s18_mapped["canonical_franchise_id"].notna().sum())
print("Unmatched records:", s18_mapped["canonical_franchise_id"].isna().sum())
print("Unique franchise-season keys:",
      s18_mapped[["canonical_franchise_id", "season"]]
      .drop_duplicates().shape[0])

S18 MAPPING VALIDATION PASSED
S18 records: 718
Mapped records: 718
Unmatched records: 0
Unique franchise-season keys: 718


In [5]:
# Load C01 season-level financial rules.
c01 = pd.read_csv(
    PROJECT_ROOT / "data/canonical/seasons.csv"
)

# Join S18 to the C01 luxury-tax threshold.
tax_check = s18_mapped.merge(
    c01[["season", "Luxury Tax"]].rename(
        columns={"Luxury Tax": "c01_tax_threshold"}
    ),
    on="season",
    how="left",
    validate="many_to_one"
)

# Compare only records where S18 provides a threshold.
has_threshold = tax_check["tax_threshold"].notna()

tax_check["threshold_difference"] = (
    tax_check["tax_threshold"] - tax_check["c01_tax_threshold"]
)

mismatches = tax_check[
    has_threshold &
    (tax_check["threshold_difference"].abs() > 1)
]

print("S18 rows:", len(tax_check))
print("S18 thresholds available:", has_threshold.sum())
print("S18 thresholds missing:", (~has_threshold).sum())
print("Threshold mismatches:", len(mismatches))

print("\nTHRESHOLD MISMATCHES BY SEASON")
display(
    mismatches.groupby("season")
    .agg(
        records=("season", "size"),
        s18_threshold=("tax_threshold", "first"),
        c01_threshold=("c01_tax_threshold", "first")
    )
)

print("\nTAX SPACE CONSISTENCY")

# Verify whether tax_space = tax_threshold - taxable_payroll.
space_check = tax_check.dropna(
    subset=["tax_threshold", "taxable_payroll", "tax_space"]
).copy()

space_check["calculated_tax_space"] = (
    space_check["tax_threshold"] - space_check["taxable_payroll"]
)

space_check["difference"] = (
    space_check["tax_space"] - space_check["calculated_tax_space"]
)

print("Comparable records:", len(space_check))
print("Tax-space mismatches (> $1):",
      (space_check["difference"].abs() > 1).sum())

display(
    space_check.loc[
        space_check["difference"].abs() > 1,
        [
            "franchise_id_x",
            "season",
            "tax_threshold",
            "taxable_payroll",
            "tax_space",
            "calculated_tax_space",
            "difference"
        ]
    ].head(10)
)

S18 rows: 718
S18 thresholds available: 688
S18 thresholds missing: 30
Threshold mismatches: 208

THRESHOLD MISMATCHES BY SEASON


,records,s18_threshold,c01_threshold
season,,,
2002-2003,29,52900000.0,52880000.0
2003-2004,29,54600000.0,54556722.0
2006-2007,30,65400000.0,65420000.0
2007-2008,30,67900000.0,67865000.0
2008-2009,30,71200000.0,71150000.0
2009-2010,30,69900000.0,69920000.0
2010-2011,30,70300000.0,70307000.0



TAX SPACE CONSISTENCY
Comparable records: 450
Tax-space mismatches (> $1): 0


KeyError: "['franchise_id_x'] not in index"

In [6]:
# --------------------------------------------------
# S18 → C04 FINAL INTEGRATION
# --------------------------------------------------

from pathlib import Path
import pandas as pd

CANONICAL_DIR = PROJECT_ROOT / "data/canonical"

# Start from the existing canonical team-season table.
c04_original = pd.read_csv(
    CANONICAL_DIR / "team_seasons.csv"
)

# Select the S18 financial fields we want to preserve.
s18_fields = [
    "canonical_franchise_id",
    "season",
    "tax_threshold",
    "taxable_payroll",
    "tax_space",
    "tax_paid",
    "estimated_tax_bill",
    "total_estimated_cost",
    "active_salary",
    "dead_salary",
    "source_file"
]

s18_for_merge = s18_mapped[s18_fields].copy()

# Rename fields to clearly identify their source.
s18_for_merge = s18_for_merge.rename(
    columns={
        "canonical_franchise_id": "franchise_id",
        "tax_threshold": "s18_tax_threshold",
        "source_file": "s18_source_file"
    }
)

# Source coverage indicator.
s18_for_merge["has_luxury_tax_source"] = True

# Ensure unique join keys.
assert not s18_for_merge.duplicated(
    ["franchise_id", "season"]
).any()

assert not c04_original.duplicated(
    ["franchise_id", "season"]
).any()

# Merge on franchise-season.
c04_enriched = c04_original.merge(
    s18_for_merge,
    on=["franchise_id", "season"],
    how="left",
    validate="one_to_one",
    indicator=True
)

# Preserve the difference between missing S18 data
# and explicitly recorded zero tax payments.
c04_enriched["has_luxury_tax_source"] = (
    c04_enriched["has_luxury_tax_source"].fillna(False)
    .astype(bool)
)

# --------------------------------------------------
# VALIDATION
# --------------------------------------------------

assert len(c04_enriched) == len(c04_original)

assert not c04_enriched.duplicated(
    ["franchise_id", "season"]
).any()

# Verify every S18 row matched C04.
assert (
    (c04_enriched["_merge"] == "both").sum()
    == len(s18_for_merge)
)

print("C04 original rows:", len(c04_original))
print("C04 enriched rows:", len(c04_enriched))

print("S18 matched records:",
      (c04_enriched["_merge"] == "both").sum())

print("C04 rows without S18:",
      (c04_enriched["_merge"] == "left_only").sum())

print("\nS18 FIELD COVERAGE")
display(
    c04_enriched[
        [
            "taxable_payroll",
            "tax_paid",
            "estimated_tax_bill",
            "active_salary",
            "dead_salary"
        ]
    ].notna().sum()
)

# Remove the merge diagnostic column.
c04_enriched = c04_enriched.drop(columns="_merge")

# Export enriched canonical C04.
c04_enriched.to_csv(
    CANONICAL_DIR / "team_seasons.csv",
    index=False
)

print("\nC04 ENRICHMENT VALIDATION PASSED")
print("Updated team_seasons.csv exported.")

C04 original rows: 776
C04 enriched rows: 776
S18 matched records: 718
C04 rows without S18: 58

S18 FIELD COVERAGE


taxable_payroll       450
tax_paid              268
estimated_tax_bill     94
active_salary         450
dead_salary           416
dtype: int64


C04 ENRICHMENT VALIDATION PASSED
Updated team_seasons.csv exported.
